# 01 — Load the PSID extract, give the variables readable names, fetch the CPI

**Reads:** `data/raw/J366383.xlsx` (the PSID Data Center extract; see README for how to rebuild it),
`data/raw/psid_variable_crosswalk.csv`, FRED (`CPIAUCSL`).

**Writes:** `data/processed/psid_wide.parquet` (one row per person, readable column names),
`data/raw/cpi_u_annual.csv`.

The extract comes out of the Data Center with PSID variable codes as column names (`ER30001`, `V17533`, …).
The crosswalk maps each code to a readable name of the form `concept_year`, e.g. `fam_income_1990`.
Nothing is reshaped here — that is notebook 02's job — this notebook only renames, caches and fetches the deflator.

In [ ]:
from pathlib import Path
import io, json, datetime as dt
import pandas as pd
import requests

ROOT = Path.cwd().parent           # notebooks run from code/
RAW = ROOT / "data" / "raw"
PROC = ROOT / "data" / "processed"
PROC.mkdir(parents=True, exist_ok=True)

EXTRACT = RAW / "J366383.xlsx"     # PSID Data Center job number; not committed (PSID terms of use)

## 1. Crosswalk: PSID code → readable name

`table` says which PSID file the variable comes from (`individual` or `family`); `year` is the survey wave
(`all` for the fixed, time-invariant identifiers). The 1968 interview number `ER30001` is both half of the
permanent person ID and the 1968 family interview number, so it appears once and is copied below.

In [ ]:
cw = pd.read_csv(RAW / "psid_variable_crosswalk.csv")
rename = dict(zip(cw["variable"], cw["name"]))
print(len(rename), "variables in the crosswalk")
cw.groupby(["table", "concept"]).size()

## 2. Read the extract (slow once, then cached)

`openpyxl` fails on this file (a non-standard sheet attribute), so the Excel is read with the `calamine` engine.
The ~35-second read happens only the first time; afterwards the parquet cache is used.

In [ ]:
if (PROC / "psid_wide.parquet").exists():
    wide = pd.read_parquet(PROC / "psid_wide.parquet")
    print("loaded cache", wide.shape)
else:
    raw = pd.read_excel(EXTRACT, engine="calamine")
    print("raw extract", raw.shape)

    extra = [c for c in raw.columns if c not in rename]       # Data Center bookkeeping columns
    missing = [c for c in rename if c not in raw.columns]
    print(len(extra), "columns not in the crosswalk (dropped):", extra[:6], "...")
    assert not missing, f"crosswalk variables missing from the extract: {missing}"

    wide = raw.drop(columns=extra).rename(columns=rename)
    wide["interview_1968"] = wide["id_1968"]                    # 1968: family interview number == 1968 ID
    wide.to_parquet(PROC / "psid_wide.parquet", index=False)
    print("saved", wide.shape)

wide.iloc[:5, :12]

In [ ]:
# Quick sanity checks on one wave: the person's interview number is the family-record key
chk = wide[wide["interview_1990"] > 0]
print("people interviewed in 1990:", len(chk))
print("interview number == family interview number:", (chk["interview_1990"] == chk["fam_interview_1990"]).mean())
chk["fam_income_1990"].describe()

## 3. CPI-U, annual average, for deflating income

PSID total family income reported in survey year *t* is income for calendar year *t − 1* (check the codebook entry
for any `fam_income_*` variable). Deflate with the CPI-U annual average of the **income year**, not the survey year.
`CPIAUCSL` is monthly; the annual average is the mean of the twelve months. Base year chosen below.

In [ ]:
BASE_YEAR = 2023

url = "https://fred.stlouisfed.org/graph/fredgraph.csv?id=CPIAUCSL"
cpi_m = pd.read_csv(io.StringIO(requests.get(url, timeout=60).text), parse_dates=["observation_date"])
cpi_m.columns = ["date", "cpi"]
cpi = (cpi_m.assign(year=cpi_m["date"].dt.year)
            .groupby("year", as_index=False)["cpi"].mean()
            .query("year >= 1967"))
cpi["deflator_to_base"] = cpi.loc[cpi["year"] == BASE_YEAR, "cpi"].iloc[0] / cpi["cpi"]
cpi.to_csv(RAW / "cpi_u_annual.csv", index=False)

json.dump({"CPIAUCSL": {"source": "FRED, U.S. Bureau of Labor Statistics, CPI-U all items, seasonally adjusted (CPIAUCSL)",
                        "url": url, "accessed": dt.date.today().isoformat(), "base_year": BASE_YEAR}},
          open(RAW / "fetch_log.json", "w"), indent=2)
cpi.tail()